# Advanced ML: Gradient Boosting with XGBoost In-Depth

XGBoost (eXtreme Gradient Boosting) is one of the most popular and efficient algorithms for structured (tabular) data. It often outperforms other algorithms like Random Forests.


# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


In [ ]:

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")
    !pip install -q xgboost torch seaborn pandas numpy matplotlib
except ImportError:
    IN_COLAB = False
    print("Running locally. Assuming dependencies are installed.")


In [ ]:

import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Generate Synthetic Data (e.g., Voter Turnout Prediction)
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'Age': np.random.randint(18, 90, n),
    'Income': np.random.normal(50000, 15000, n),
    'Education_Level': np.random.randint(1, 5, n), # 1=HS, 2=BA, etc.
    'Interest_in_Politics': np.random.randint(1, 10, n),
    'Distance_to_Polls': np.random.exponential(5, n)
})

# Target: Voted (1) or Not (0)
# Complex relationship
logits = (df['Age'] * 0.05 + 
          df['Interest_in_Politics'] * 0.8 + 
          df['Education_Level'] * 0.5 - 
          df['Distance_to_Polls'] * 0.2 - 5)
probs = 1 / (1 + np.exp(-logits))
df['Voted'] = (np.random.rand(n) < probs).astype(int)

print(df['Voted'].value_counts())


## 2. Train/Test Split

In [ ]:

X = df.drop('Voted', axis=1)
y = df['Voted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 3. Train XGBoost Classifier

In [ ]:

model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, learning_rate=0.1)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))


## 4. Feature Importance

In [ ]:

xgb.plot_importance(model, importance_type='weight')
plt.title('Feature Importance (Weight)')
plt.show()
